# Support Ticket Router: Your First LLM Inside a Graph

A small LangGraph workflow that uses one structured LLM decision to route each support ticket. The six clear examples check the three routes; the mixed example is left for the model to decide.

In [ ]:
!pip -q install langgraph langchain-openai

## Load the API key from Colab Secrets

In Colab, add a secret named `OPENAI_API_KEY` and enable notebook access. The key is read from Colab and is never displayed or stored in this notebook.

In [ ]:
import os
from google.colab import userdata

os.environ["OPENAI_API_KEY"] = userdata.get("OPENAI_API_KEY")

## Define the state and structured classification

The model returns exactly one category. Billing is for payment, invoice, refund, or subscription issues; technical is for bugs, crashes, error messages, and login problems; general is for feature ideas, general questions, and thanks. The model should use the underlying issue rather than a single keyword when a ticket mentions more than one topic.

In [ ]:
from typing import Literal, TypedDict

from langchain_openai import ChatOpenAI
from pydantic import BaseModel, Field
from langgraph.graph import END, START, StateGraph

MODEL_NAME = "gpt-6-luna"

class TicketState(TypedDict):
    ticket: str
    category: Literal["billing", "technical", "general"]
    team: str
    reply: str

class TicketCategory(BaseModel):
    category: Literal["billing", "technical", "general"] = Field(
        description="The single best support category for the ticket."
    )

classifier = ChatOpenAI(model=MODEL_NAME, temperature=0).with_structured_output(TicketCategory)

## Build the graph

`classify` is the only node that calls the model. Its output is stored in state. The routing function only reads that state, and the three team nodes return fixed replies without another model call.

In [ ]:
CLASSIFICATION_INSTRUCTIONS = (
    "Classify the customer's underlying support issue. Use 'billing' for charges, payments, invoices, refunds, or subscriptions; "
    "use 'technical' for bugs, crashes, error messages, or login problems; use 'general' for feature ideas, general questions, "
    "or thank-you notes. Decide from the meaning and main problem, not from an isolated keyword. For mixed tickets, choose the "
    "category that best matches what the customer needs help resolving. Return only the structured category."
)

def classify(state: TicketState) -> dict:
    response = classifier.invoke(
        [
            ("system", CLASSIFICATION_INSTRUCTIONS),
            ("human", state["ticket"]),
        ]
    )
    return {"category": response.category}

def route(state: TicketState) -> str:
    return state["category"]

def billing_team(state: TicketState) -> dict:
    return {"team": "Billing", "reply": "Your ticket has been forwarded to Billing."}

def tech_team(state: TicketState) -> dict:
    return {"team": "Tech", "reply": "Your ticket has been forwarded to Tech."}

def general_team(state: TicketState) -> dict:
    return {"team": "General", "reply": "Your ticket has been forwarded to General."}

builder = StateGraph(TicketState)
builder.add_node("classify", classify)
builder.add_node("billing_team", billing_team)
builder.add_node("tech_team", tech_team)
builder.add_node("general_team", general_team)
builder.add_edge(START, "classify")
builder.add_conditional_edges(
    "classify",
    route,
    {"billing": "billing_team", "technical": "tech_team", "general": "general_team"},
)
builder.add_edge("billing_team", END)
builder.add_edge("tech_team", END)
builder.add_edge("general_team", END)

graph = builder.compile()

## Visualize the graph

In [ ]:
from IPython.display import Image, display

display(Image(graph.get_graph().draw_mermaid_png()))

## Invoke and test the router

Each ticket makes one model call. The first call uses the required invocation shape; the remaining examples cover two clear tickets per team and one ambiguous ticket. The ambiguous result is observed rather than forced to a predetermined category.

In [ ]:
result = graph.invoke({"ticket": "I was charged twice this month"})

ticket_cases = [
    ("billing", "How can I download last month's invoice?"),
    ("technical", "The app crashes whenever I open my task list."),
    ("technical", "I cannot log in because the app returns an error."),
    ("general", "Could you add a calendar view for tasks?"),
    ("general", "Thank you, this app makes planning much easier."),
    (None, "The app crashes every time I try to pay."),
]

test_results = [("billing", "I was charged twice this month", result)]
for expected_category, ticket in ticket_cases:
    test_results.append((expected_category, ticket, graph.invoke({"ticket": ticket})))

required_keys = {"ticket", "category", "team", "reply"}
for expected_category, ticket, output in test_results:
    assert required_keys.issubset(output), f"Missing result keys for: {ticket}"
    assert output["ticket"] == ticket, f"Original ticket was not preserved: {ticket}"
    assert output["category"] in {"billing", "technical", "general"}
    assert output["team"] in {"Billing", "Tech", "General"}
    assert isinstance(output["reply"], str) and output["reply"]
    if expected_category is not None:
        assert output["category"] == expected_category, (
            f"Expected {expected_category} but got {output['category']} for: {ticket}"
        )
    print(f"Ticket: {ticket}\nCategory: {output['category']}\nTeam: {output['team']}\nReply: {output['reply']}\n")

clear_categories = {expected for expected, _, _ in test_results if expected is not None}
assert clear_categories == {"billing", "technical", "general"}
print("All six clear cases matched their expected categories; the ambiguous case was left to the model.")